In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Set matplotlib style for better notebook visuals
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
class UCSDVideoDataset(Dataset):
    """
    Custom PyTorch Dataset for UCSD Pedestrian dataset.
    Assumes data is structured as: data/UCSD_Anomaly_Dataset/UCSDped2/Train/Train001/001.tif
    """
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.frames = []

        # Traverse directory to find all video frame folders
        if os.path.exists(root_dir):
            for folder in sorted(os.listdir(root_dir)):
                folder_path = os.path.join(root_dir, folder)
                if os.path.isdir(folder_path):
                    for file in sorted(os.listdir(folder_path)):
                        if file.endswith(('.tif', '.png', '.jpg')):
                            self.frames.append(os.path.join(folder_path, file))
        else:
            print(f"Warning: Directory {root_dir} not found. Ensure download_ucsd.sh has run successfully.")

    def __len__(self):
        return len(self.frames)

    def __getitem__(self, idx):
        img_path = self.frames[idx]

        # Read image using OpenCV
        image = cv2.imread(img_path)
        if image is not None:
            # Convert BGR to RGB for standard PyTorch/Matplotlib pipelines
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            image = self.transform(image)

        return image, img_path

In [ ]:
# Define standard transformations for your Backbone (YOLO/ViT)
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)), # Adjust to 640x640 if specifically testing YOLOv8 natively
    transforms.ToTensor()
])

# Update this path relative to your notebooks/ directory
dataset_path = "/kaggle/input/datasets/orvile/ucsd-anomaly-dataset/UCSD_Anomaly_Dataset.v1p2/UCSDped2/Train"

ucsd_dataset = UCSDVideoDataset(root_dir=dataset_path, transform=transform)

# Create DataLoader for batching
data_loader = DataLoader(ucsd_dataset, batch_size=4, shuffle=True, num_workers=2)

print(f"Total frames loaded: {len(ucsd_dataset)}")

In [ ]:
def visualize_batch(dataloader):
    """Fetches a batch from the DataLoader and visualizes it."""
    # Get one batch of data
    dataiter = iter(dataloader)
    try:
        images, paths = next(dataiter)
    except StopIteration:
        print("DataLoader is empty. Please check your data paths.")
        return

    # Plot the images
    fig, axes = plt.subplots(1, len(images), figsize=(16, 4))

    # Handle case where batch_size might be 1
    if len(images) == 1:
        axes = [axes]

    for i in range(len(images)):
        # Convert PyTorch tensor (C, H, W) back to numpy (H, W, C) for Matplotlib
        img_np = images[i].numpy().transpose((1, 2, 0))
        axes[i].imshow(img_np)

        # Extract folder and filename for the title (e.g., "Train001/050.tif")
        title = os.path.join(*os.path.normpath(paths[i]).split(os.sep)[-2:])
        axes[i].set_title(title, fontsize=10)
        axes[i].axis('off')

    plt.suptitle("Batch Visualization - UCSD Pedestrian Frames", fontsize=14)
    plt.tight_layout()
    plt.show()

# Execute the visualization
visualize_batch(data_loader)